In [1]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Annotated, List

import random

import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.grammar import extract_grammar
from geneticengine.grammar.decorators import weight
from geneticengine.problems import SingleObjectiveProblem, MultiObjectiveProblem
from geneticengine.random.sources import NativeRandomSource
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.evaluation.budget import TimeBudget, EvaluationBudget
from geneticengine.representations.tree.initializations import MaxDepthDecider, FullDecider, ProgressivelyTerminalDecider, PositionIndependentGrowDecider
from geneticengine.representations.tree.operators import GrowInitializer, PositionIndependentGrowInitializer, FullInitializer, RampedHalfAndHalfInitializer
from geneticengine.algorithms.gp.operators.initializers import HalfAndHalfInitializer, StandardInitializer
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.representations.grammatical_evolution.structured_ge import StructuredGrammaticalEvolutionRepresentation
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.evaluation.parallel import ParallelEvaluator

from geneticengine.algorithms.gp.operators.combinators import ParallelStep, SequenceStep
from geneticengine.algorithms.gp.operators.crossover import GenericCrossoverStep
from geneticengine.algorithms.gp.operators.elitism import ElitismStep
from geneticengine.algorithms.gp.operators.mutation import GenericMutationStep
from geneticengine.algorithms.gp.operators.novelty import NoveltyStep
from geneticengine.algorithms.gp.operators.selection import LexicaseSelection, TournamentSelection

from geneticengine.solutions.individual import Individual, PhenotypicIndividual
from geneticengine.algorithms.gp.structure import GeneticStep
from geneticengine.problems import Problem
from geneticengine.random.sources import RandomSource
from geneticengine.representations.api import RepresentationWithCrossover, Representation
from geneticengine.evaluation import Evaluator
from typing import Iterator, Any, TypeVar

from sklearn.datasets import load_breast_cancer

import pandas as pd

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

import time

import seaborn as sns
import matplotlib.pyplot as plt

import os     

from functools import lru_cache

import lightgbm as lgb

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
target_fpr_value = 0.05

In [4]:
df_orig = pd.read_csv('Variant V.csv')

df_orig.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 34 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   fraud_bool                        1000000 non-null  int64  
 1   income                            1000000 non-null  float64
 2   name_email_similarity             1000000 non-null  float64
 3   prev_address_months_count         1000000 non-null  int64  
 4   current_address_months_count      1000000 non-null  int64  
 5   customer_age                      1000000 non-null  int64  
 6   days_since_request                1000000 non-null  float64
 7   intended_balcon_amount            1000000 non-null  float64
 8   payment_type                      1000000 non-null  object 
 9   zip_count_4w                      1000000 non-null  int64  
 10  velocity_6h                       1000000 non-null  float64
 11  velocity_24h                      1000

In [5]:
df = df_orig.copy()
X = df.drop(['fraud_bool'], axis=1)
y = df['fraud_bool']

X_train = X[df['month'] < 6]
y_train = y[df['month'] < 6]
X_test = X[df['month'] >= 6]
y_test = y[df['month'] >= 6]

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

#drop month column
X_train.drop('month', axis=1, inplace=True)
X_val.drop('month', axis=1, inplace=True)
X_test.drop('month', axis=1, inplace=True)

In [6]:
categorical_features = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]


encoders = {}
for feat in categorical_features:
    encoder = LabelEncoder()
    X_train[feat] = encoder.fit_transform(X_train[feat])
    X_val[feat] = encoder.transform(X_val[feat])
    X_test[feat] = encoder.transform(X_test[feat])
    encoders[feat] = encoder

In [7]:
print(y_train.value_counts(),y_val.value_counts(), y_test.value_counts())

fraud_bool
0    599960
1      6702
Name: count, dtype: int64 fraud_bool
0    149990
1      1676
Name: count, dtype: int64 fraud_bool
0    239020
1      2652
Name: count, dtype: int64


In [8]:
feature_names = X_train.columns.tolist()
n_features = len(feature_names)

In [9]:
model_baseline = lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.02, num_leaves=30, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)

model_baseline.fit(X_train, y_train)

train_probs = model_baseline.predict_proba(X_train)[:,1]

fpr, tpr, thresholds = roc_curve(y_train, train_probs)

target_fpr = target_fpr_value

if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    train_tpr_at_fpr = tpr[best_index]

val_probs = model_baseline.predict_proba(X_val)[:,1]

fpr, tpr, thresholds = roc_curve(y_val, val_probs)

baseline_tpr_at_fpr = 0.0

if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    baseline_tpr_at_fpr = tpr[best_index]

print(f"Train TPR: {train_tpr_at_fpr}, Validation TPR: {baseline_tpr_at_fpr}")

Train TPR: 0.7499253954043569, Validation TPR: 0.736873508353222


In [10]:
@dataclass
class Value(ABC):
    def evaluate(self):
        pass

class Scalar(ABC):
    pass

@weight(0.4)
@dataclass #Scalar Features (1)
class ScalarVar(Scalar): 
    index: Annotated[int, IntRange(0,n_features-1)]

    def evaluate(self, X_np):
        return X_np[:, self.index]
    
    def __str__(self):
        return feature_names[self.index]
    
#scalar -> scalar
@weight(0.2)
@dataclass 
class Add(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return self.left.evaluate(X_np) + self.right.evaluate(X_np)
    
    def __str__(self):
        return f"({self.left} + {self.right})"

@weight(0.2)
@dataclass
class Subtract(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return (self.left.evaluate(X_np)) - (self.right.evaluate(X_np))
    
    def __str__(self):
        return f"({self.left} - {self.right})"

@weight(0.2)
@dataclass
class Multiply(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return self.left.evaluate(X_np) * self.right.evaluate(X_np)
    
    def __str__(self):
        return f"({self.left} * {self.right})"

In [11]:
grammar = extract_grammar([Add, Subtract, Multiply,ScalarVar], Scalar)
print(f"Grammar: {repr(grammar)}")

Grammar: Grammar<Starting=Scalar,Productions={
Scalar -> Add(right: Scalar, left: Scalar)<0.20>|
	Subtract(right: Scalar, left: Scalar)<0.20>|
	Multiply(right: Scalar, left: Scalar)<0.20>|
	ScalarVar(index: Annotated[int])<0.40>
}


In [12]:
ARCHIVE_TRAIN_DF = X_train.copy()
ARCHIVE_VAL_DF = X_val.copy()

ARCHIVE_TEMP : list[Individual] = []
ARCHIVE_IND : list[Individual] = []

In [13]:
X_train_np = ARCHIVE_TRAIN_DF.to_numpy()
X_val_np = ARCHIVE_VAL_DF.to_numpy()

def fitness_function(individual: Scalar): #individual -> expression
    
    if str(individual) in ARCHIVE_TRAIN_DF.columns:
        return [0.0, 0.0, 1000.0, 1000.0]
    
    start = time.perf_counter()
    train_feature = individual.evaluate(X_train_np)
    test_feature = individual.evaluate(X_val_np)
    if train_feature.ndim == 0:
        train_feature = np.full(X_train_np.shape[0], train_feature)
    if test_feature.ndim == 0:
        test_feature = np.full(X_val_np.shape[0], test_feature)

    X_train_augmented = np.c_[X_train_np, np.array(train_feature).reshape(-1,1)]
    X_val_augmented = np.c_[X_val_np, np.array(test_feature).reshape(-1,1)]

    model = lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.02, num_leaves=30, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)

    model.fit(X_train_augmented, y_train)
    # print(model.feature_names_in_)
    probs = model.predict_proba(X_val_augmented)[:, 1]

    fpr, tpr, thresholds = roc_curve(y_val, probs)
    tpr_at_fpr = 0.0
    if np.any(fpr <= target_fpr):
        valid_indices = np.where(fpr<=target_fpr)[0]
        best_indice = valid_indices[np.argmax(tpr[valid_indices])]
        tpr_at_fpr = tpr[best_indice]

    tpr_diff = tpr_at_fpr-baseline_tpr_at_fpr
        
    features, num_operations = analyse_complexity(individual)

    end = time.perf_counter()
    elapsed = end - start
    return [tpr_at_fpr, tpr_diff, num_operations, elapsed]



In [14]:
def analyse_complexity(individual: Scalar):
    if isinstance(individual, ScalarVar):
        return {individual.index}, 0 #unique feature
    
    total_features = set()
    total_operations = 1
    if hasattr(individual, 'left') and hasattr(individual, 'right'):
        left_features, left_operations = analyse_complexity(individual.left)
        right_features, right_operations = analyse_complexity(individual.right)
        total_features.update(left_features)
        total_features.update(right_features)
        total_operations += left_operations + right_operations
    elif hasattr(individual, 'arr'):
        arr_features, arr_operations = analyse_complexity(individual.arr)
        total_features.update(arr_features)
        total_operations += arr_operations
    return total_features, total_operations


In [15]:
class ArchiveStep(GeneticStep):
    def iterate(
        self,
        problem: Problem,
        evaluator: Evaluator,
        representation: Representation,
        random: RandomSource,
        population: Iterator[PhenotypicIndividual],
        target_size: int,
        generation: int,
    ) -> Iterator[PhenotypicIndividual]:
        global ARCHIVE_TEMP, train_tpr_at_fpr, baseline_tpr_at_fpr, ARCHIVE_TRAIN_DF, ARCHIVE_VAL_DF, X_train_np, X_val_np, ARCHIVE_IND
        best_fitness = 0
        for i, individual in enumerate(population):
            if individual.get_fitness(problem).fitness_components[0] > baseline_tpr_at_fpr and individual.get_fitness(problem).fitness_components[0] > best_fitness:
                best_fitness = individual.get_fitness(problem).fitness_components[0]
                print("New Individual:", str(individual.get_phenotype()), "Fitness:", individual.get_fitness(problem).fitness_components)
                if not ARCHIVE_TEMP:
                    ARCHIVE_TEMP.append(individual)
                else:
                    ARCHIVE_TEMP[0] = individual
            yield individual

        if ARCHIVE_TEMP:
            print(f"Archive Size: {len(ARCHIVE_TEMP)}")
            for ind in ARCHIVE_TEMP:
                ARCHIVE_IND.append(ind)
                train_feature_new = ind.get_phenotype().evaluate(X_train_np)
                val_feature_new = ind.get_phenotype().evaluate(X_val_np)

                ARCHIVE_TRAIN_DF[str(ind)] = train_feature_new
                ARCHIVE_VAL_DF[str(ind)] = val_feature_new

            ARCHIVE_TRAIN_DF = ARCHIVE_TRAIN_DF.loc[:, ~ARCHIVE_TRAIN_DF.columns.duplicated()]
            ARCHIVE_VAL_DF = ARCHIVE_VAL_DF.loc[:, ~ARCHIVE_VAL_DF.columns.duplicated()]

            ARCHIVE_TRAIN_DF.columns = [str(col) for col in ARCHIVE_TRAIN_DF.columns]
            ARCHIVE_VAL_DF.columns = [str(col) for col in ARCHIVE_VAL_DF.columns]

            model = lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.02, num_leaves=30, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)
            model.fit(ARCHIVE_TRAIN_DF, y_train)
            target_fpr = target_fpr_value
            probs = model.predict_proba(ARCHIVE_VAL_DF)[:,1]
            fpr, tpr, thresholds = roc_curve(y_val, probs)
            if np.any(fpr <= target_fpr):
                valid_indices = np.where(fpr <= target_fpr)[0]
                best_index = valid_indices[np.argmax(tpr[valid_indices])]
                baseline_tpr_at_fpr = tpr[best_index]
            print(f"Validation TPR: {baseline_tpr_at_fpr}, Validation shape: {ARCHIVE_VAL_DF.shape}")
            ARCHIVE_TEMP = []
            X_train_np = ARCHIVE_TRAIN_DF.to_numpy()
            X_val_np = ARCHIVE_VAL_DF.to_numpy()

In [16]:
def lexicase_step():
    return SequenceStep(
        ArchiveStep(),
        ParallelStep(
            [
                ElitismStep(),
                NoveltyStep(),
                SequenceStep(
                    LexicaseSelection(epsilon=True),
                    # TournamentSelection(tournament_size=3),
                    GenericCrossoverStep(0.9),
                    GenericMutationStep(0.1),
                )
            ],
            weights=[0.05, 0.05, 0.9]
        ),
    )

prob = MultiObjectiveProblem(
    fitness_function=fitness_function,
    minimize=[False, False, True, True],
)
r = NativeRandomSource(123)
alg = GeneticProgramming(
    problem=prob,
    budget=TimeBudget(600),
    population_size=50,
    representation=TreeBasedRepresentation(grammar, MaxDepthDecider(r, grammar, 4)),
    random=r,
    step=lexicase_step(),
    tracker=ProgressTracker(
        prob,
        recorders=[CSVSearchRecorder(
            csv_path='output_variant5.csv', 
            problem=prob, 
            fields={
                    "Eval Time": lambda t,i,p: i.get_fitness(p).fitness_components[3],
                    "TPR Test": lambda t,i,p: i.get_fitness(p).fitness_components[0],
                    "TPR Test Diff": lambda t,i,p: i.get_fitness(p).fitness_components[1],
                    "Expression": lambda t, i, p: i.get_phenotype(),
                    "Num Operations": lambda t,i,p: i.get_fitness(p).fitness_components[2],
                    'Generation': lambda t,i,p: i.metadata["generation"]
                    },
            only_record_best_individuals=False)]
    )
    
)

solutions = alg.search()

New Individual: (has_other_cards + ((prev_address_months_count + email_is_free) * (session_length_in_minutes + source))) Fitness: [np.float64(0.7374701670644391), np.float64(0.0005966587112171684), 4, 2.934319000050891]
New Individual: (current_address_months_count * payment_type) Fitness: [np.float64(0.7380668257756563), np.float64(0.0011933174224343368), 1, 3.2431913999607787]
Archive Size: 1
Validation TPR: 0.7380668257756563, Validation shape: (151666, 33)
New Individual: (((bank_months_count * phone_home_valid) * zip_count_4w) * foreign_request) Fitness: [np.float64(0.7386634844868735), np.float64(0.0005966587112171684), 3, 2.882738000014797]
New Individual: ((name_email_similarity + (phone_mobile_valid - device_distinct_emails_8w)) + ((foreign_request * keep_alive_session) * bank_branch_count_8w)) Fitness: [np.float64(0.7392601431980907), np.float64(0.0011933174224344478), 5, 2.425015499989968]
New Individual: ((income * (zip_count_4w - payment_type)) * customer_age) Fitness: [np

In [19]:
model = lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.02, num_leaves=30, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)
model.fit(X_train, y_train)
original_probs = model.predict_proba(X_test)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, original_probs)
target_fpr = target_fpr_value
if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    original_tpr_at_fpr = tpr[best_index]
print(f"Original Test TPR at FPR {target_fpr}: {original_tpr_at_fpr}")

#train on the augmented set
X_test_enhanced = X_test.copy()
for ind in ARCHIVE_IND:
    X_test_enhanced[str(ind.get_phenotype())] = ind.get_phenotype().evaluate(X_test.to_numpy())
model = lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.02, num_leaves=30, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)
model.fit(ARCHIVE_TRAIN_DF, y_train)
augmented_probs = model.predict_proba(X_test_enhanced)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, augmented_probs)
target_fpr = target_fpr_value
if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    augmented_tpr_at_fpr = tpr[best_index]
print(f"Augmented Test TPR at FPR {target_fpr}: {augmented_tpr_at_fpr}")
print(f"Improvement in TPR at FPR {target_fpr}: {augmented_tpr_at_fpr - original_tpr_at_fpr}")

Original Test TPR at FPR 0.05: 0.34841628959276016
Augmented Test TPR at FPR 0.05: 0.3469079939668175
Improvement in TPR at FPR 0.05: -0.0015082956259426794
